## 9.4 — stack ו-concatenate — לאחד ריצות

בסעיף 9.1 בנינו את `Y` בלולאה מפורשת. בפועל, כשיש כבר כמה מערכים חד-ממדיים נפרדים (למשל, ריצה שנמדדה בנפרד בכל פעם), משלבים אותם עם:

- **`np.stack([a1, a2, ...])`** — יוצר **ציר חדש** ומניח את כל המערכים זה לצד זה עליו. אם כל `a_i` בצורה `(60,)`, ו-5 מהם, `np.stack` נותן `(5, 60)`.
- **`np.concatenate([a1, a2, ...])`** — מחבר לאורך ציר **קיים**, בלי ליצור ציר חדש. שני מערכים בצורה `(60,)` נותנים מערך אחד בצורה `(120,)`.

ההבדל הזה בדיוק: `stack` מוסיף ממד, `concatenate` לא.

In [ ]:
import numpy as np

g = 9.8
theta = np.radians(45)
v0_nom = 20.0

def make_run(v0, n_t=60):
    t_flight = 2 * v0_nom * np.sin(theta) / g
    t = np.linspace(0, t_flight, n_t)
    y = v0 * np.sin(theta) * t - 0.5 * g * t**2
    return t, y

### דוגמה: איחוד ריצות נפרדות ל-`stack`

נניח שכל ריצה נמדדה בנפרד (כמו קובץ נתונים משלה), וקיבלנו 5 מערכי גובה נפרדים. `np.stack` מאחד אותם למערך "ריצה × זמן" אחד.

In [ ]:
rng = np.random.default_rng(0)
v0_runs = rng.normal(v0_nom, 1.0, size=5)

t, y0 = make_run(v0_runs[0])
_, y1 = make_run(v0_runs[1])
_, y2 = make_run(v0_runs[2])
_, y3 = make_run(v0_runs[3])
_, y4 = make_run(v0_runs[4])

Y = np.stack([y0, y1, y2, y3, y4])
print("Y.shape:", Y.shape)   # (5, 60)

### דוגמה: הוספת ריצה חדשה עם `concatenate`

בוצעה ריצה שישית. במקום לבנות הכול מחדש, מוסיפים אותה לאורך ציר הריצות הקיים.

In [ ]:
v0_new = rng.normal(v0_nom, 1.0)
_, y_new = make_run(v0_new)

Y_extended = np.concatenate([Y, y_new[None, :]])   # y_new[None, :] הופך אותו לצורה (1, 60) - נעמיק ב-None בסעיף 9.6
print("Y_extended.shape:", Y_extended.shape)   # (6, 60)

### באג נפוץ: `stack` במקום `concatenate` (או להפך)

`np.stack([Y, y_new])` **לא** יעבוד ישירות כאן — `Y` בצורה `(5, 60)` ו-`y_new` בצורה `(60,)` אינם "אותה צורה", וזו בדיוק הדרישה של `stack`. השגיאה כאן **כן** מופיעה (`ValueError`), בניגוד לבאגי הצירים השקטים שראינו — אבל כדאי לזהות אותה מייד: הפתרון הוא `concatenate` עם `[None, :]`, לא `stack`.

### נסו בעצמכם

בנו מערך `XY` בצורה `(5, 60, 2)` (ריצה, זמן, קואורדינטה) מתוך `Y` (הגבהים) ומתוך מערך `X` דומה שתבנו לבד לקואורדינטת ה-$x$, בעזרת `np.stack` עם `axis=-1`.

In [ ]:
# X = np.stack([...])  # בנו בדומה ל-Y, עם cos במקום sin
# XY = np.stack([X, Y], axis=?)

`````{admonition} פתרון
:class: dropdown, tip
```python
xs = []
for v0_i in v0_runs:
    t_flight = 2 * v0_nom * np.sin(theta) / g
    t = np.linspace(0, t_flight, 60)
    xs.append(v0_i * np.cos(theta) * t)
X = np.stack(xs)

XY = np.stack([X, Y], axis=-1)
print(XY.shape)   # (5, 60, 2)
```
`````

### בדקו את עצמכם

In [ ]:
from jupyterquiz import display_quiz

questions = [
    {
        "question": "יש 4 מערכים בצורה <code>(60,)</code>. מה תחזיר <code>np.stack([a, b, c, d])</code>?",
        "type": "multiple_choice",
        "answers": [
            {"answer": "מערך בצורה (240,)", "correct": False, "feedback": "זו תוצאת concatenate, לא stack."},
            {"answer": "מערך בצורה (4, 60)", "correct": True, "feedback": "נכון — stack יוצר ציר חדש."},
            {"answer": "שגיאה, כי אי אפשר לחבר 4 מערכים", "correct": False, "feedback": "לא — זה בדיוק מקרה השימוש הנפוץ ביותר של stack."}
        ]
    }
]
display_quiz(questions)

### תרגול עצמי

יש לכם `Y` בצורה `(5, 60)` ורוצים לחבר אליו **שתי** ריצות נוספות בבת אחת (לא אחת בכל פעם). בנו את שתי הריצות החדשות כמערך `(2, 60)` (עם `np.stack`), ואז השתמשו ב-`np.concatenate` כדי לקבל מערך סופי בצורה `(7, 60)`.

In [ ]:
# v0_new2 = rng.normal(v0_nom, 1.0, size=2)
# ...

`````{admonition} פתרון
:class: dropdown, tip
```python
v0_new2 = rng.normal(v0_nom, 1.0, size=2)
new_runs = np.stack([make_run(v)[1] for v in v0_new2])   # (2, 60)

Y_final = np.concatenate([Y, new_runs])
print(Y_final.shape)   # (7, 60)
```
`````